# Univariate EDA

Reproducible notebook for the Phase 3 univariate analysis. It expects the Phase 1 Parquet store and the Phase 2 clean timestamp mask. Figures are written to `artifacts/univariate-eda/` and statistics to `variable_stats.parquet`.

In [ ]:
from pathlib import Path
import json
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

STORE = Path('../data/parquet')
CLEAN_MASK = Path('../artifacts/data-quality/clean-timestamps.json')
OUT = Path('../artifacts/univariate-eda')
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
con = duckdb.connect()
df = con.execute("""
  select *
  from read_parquet(? || '/source=*/year=*/month=*/*.parquet')
""", [str(STORE)]).df()
df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'], utc=True)

if CLEAN_MASK.exists():
    mask = json.loads(CLEAN_MASK.read_text())
    clean = set(mask.get('sharedCleanTimestamps', []))
    if clean:
        df = df[df['timestamp_utc'].astype(str).isin(clean)]

df.head()


In [ ]:
stats_rows = []
for (source, variable), g in df.groupby(['source', 'variable']):
    x = g['value'].dropna().astype(float)
    if len(x) < 2:
        continue
    row = {
        'source': source,
        'variable': variable,
        'stratum': 'all',
        'count': len(x),
        'mean': x.mean(),
        'median': x.median(),
        'std': x.std(),
        'skew': x.skew(),
        'kurtosis': x.kurtosis(),
        'p01': x.quantile(0.01),
        'p05': x.quantile(0.05),
        'p95': x.quantile(0.95),
        'p99': x.quantile(0.99),
    }
    stats_rows.append(row)
    fig, ax = plt.subplots(figsize=(8, 4))
    x.plot(kind='hist', bins=64, density=True, alpha=0.45, ax=ax)
    x.plot(kind='kde', ax=ax)
    ax.set_title(f'{source} / {variable}')
    fig.tight_layout()
    fig.savefig(OUT / f'{source}-{variable}-distribution.png', dpi=150)
    plt.close(fig)

stats = pd.DataFrame(stats_rows)
stats.to_parquet('../variable_stats.parquet', index=False)
stats.to_json(OUT / 'variable_stats.json', orient='records', indent=2)
stats.head()
